In [26]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [27]:
results_root = Path("results")

csv_paths = sorted(results_root.rglob("2_consecutive_rows_report.csv"))

if len(csv_paths) == 0:
    raise FileNotFoundError(
        "No files named '2_consecutive_rows_report.csv' were found under the 'results' folder."
    )

dfs = []
for p in csv_paths:
    tmp = pd.read_csv(p).copy()

    # infer subject folder name, e.g. BA1
    # path example: results/BA1/2_consecutive_rows_report.csv
    subject_id = None
    for part in p.parts:
        if str(part).upper().startswith("BA"):
            subject_id = part
            break

    tmp["subject_id"] = subject_id if subject_id is not None else p.parent.name
    tmp["source_file"] = str(p)
    dfs.append(tmp)

merged_reports = pd.concat(dfs, ignore_index=True)

print("Merged shape:", merged_reports.shape)
print("Number of source files:", len(csv_paths))
print("Subjects found:", merged_reports["subject_id"].nunique())
print(merged_reports.head())


Merged shape: (7641, 7)
Number of source files: 83
Subjects found: 83
                task_name  total_rows  kept_rows  discarded_rows  \
0           baseline_talk        1722       1722               0   
1  emotional_instructions         395        395               0   
2               eyesclose        2534       2534               0   
3                eyesopen        2571       2571               0   
4   frontal1_memorization        2359       2359               0   

   discarded_percent subject_id                                source_file  
0                0.0        BA1  results/BA1/2_consecutive_rows_report.csv  
1                0.0        BA1  results/BA1/2_consecutive_rows_report.csv  
2                0.0        BA1  results/BA1/2_consecutive_rows_report.csv  
3                0.0        BA1  results/BA1/2_consecutive_rows_report.csv  
4                0.0        BA1  results/BA1/2_consecutive_rows_report.csv  


In [28]:
stim_summary = (
    merged_reports
    .groupby("task_name", as_index=False)
    .agg(
        total_rows=("total_rows", "sum"),
        kept_rows=("kept_rows", "sum"),
        discarded_rows=("discarded_rows", "sum")
    )
)

stim_summary["discarded_percent"] = np.where(
    stim_summary["total_rows"] > 0,
    100 * stim_summary["discarded_rows"] / stim_summary["total_rows"],
    0
)

# ---------------------------------------------------------
# Load stimulus metadata and create canonical labels
# ---------------------------------------------------------
stim_meta_path = Path("stimuli_descriptions_and_expectations.csv")
stim_meta = pd.read_csv(stim_meta_path).copy()

stim_meta["original_stimuli_name"] = stim_meta["original_stimuli_name"].astype(str).str.strip()
stim_meta["cognitive_domain"] = stim_meta["cognitive_domain"].astype(str).str.strip()

# Keep only valid stimuli from metadata
allowed_stimuli = stim_meta["original_stimuli_name"].dropna().unique().tolist()

# Create labels such as:
# Motor coordination task 1
# Motor coordination task 2
stim_meta["task_number_within_domain"] = (
    stim_meta.groupby("cognitive_domain").cumcount() + 1
)

stim_meta["domain_task_label"] = (
    stim_meta["cognitive_domain"]
    + " task "
    + stim_meta["task_number_within_domain"].astype(str)
)

# Mapping: original stimulus -> readable label
stimulus_label_map = dict(
    zip(stim_meta["original_stimuli_name"], stim_meta["domain_task_label"])
)

# ---------------------------------------------------------
# Filter only stimuli that exist in metadata
# ---------------------------------------------------------
stim_summary = stim_summary[
    stim_summary["task_name"].isin(allowed_stimuli)
].copy()

# Add readable label
stim_summary["task_label"] = stim_summary["task_name"].map(stimulus_label_map)

# Preserve order from metadata file
stim_summary["task_name"] = pd.Categorical(
    stim_summary["task_name"],
    categories=allowed_stimuli,
    ordered=True
)

stim_summary = stim_summary.sort_values("task_name").reset_index(drop=True)

# Sort for the two panels
stim_kept_sorted = stim_summary.sort_values("kept_rows", ascending=False).reset_index(drop=True)
stim_disc_sorted = stim_summary.sort_values("discarded_percent", ascending=False).reset_index(drop=True)

In [29]:
'''

FIGURE 2
'''

# Load metadata
path = Path("data/PilotDatasetSubjects.xlsx")
df = pd.read_excel(path, sheet_name=0, skiprows=1)

# Basic cleaning
df = df.copy()
# -----------------------------
# Basic cleaning
# -----------------------------
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Gender"] = df["Gender"].astype(str).str.strip().str.upper()

# -----------------------------
# Basic cleaning
# -----------------------------
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Gender"] = df["Gender"].astype(str).str.strip().str.upper()
df["Education"] = df["Education"].astype(str).str.strip()

gender_map = {"F": "Female", "M": "Male"}
df["GenderLabel"] = df["Gender"].map(gender_map).fillna("Other/Unknown")

# Remove fake nan strings if present
df["Education"] = df["Education"].replace(["nan", "NaN", "None", ""], np.nan)

# -----------------------------
# Basic statistics
# -----------------------------
age = df["Age"].dropna()
mean_age = age.mean()
median_age = age.median()
min_age = age.min()
max_age = age.max()

gender_counts = df["GenderLabel"].value_counts()
female_n = int(gender_counts.get("Female", 0))
male_n = int(gender_counts.get("Male", 0))
other_n = int(gender_counts.get("Other/Unknown", 0))

education_counts = (
    df["Education"]
    .dropna()
    .value_counts()
    .sort_values(ascending=False)
)

# -----------------------------
# Style
# -----------------------------
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

blue = "#123B7A"
red = "#C62828"
gray = "#6E6E6E"

# -----------------------------
# Helper to force visible axes
# -----------------------------
def style_axis(ax):
    ax.set_facecolor("white")
    ax.tick_params(axis="both", colors="black", labelcolor="black")
    ax.xaxis.label.set_color("black")
    ax.yaxis.label.set_color("black")
    ax.title.set_color("black")
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.0)

# -----------------------------
# Create figure
# -----------------------------
fig, axes = plt.subplots(3, 2, figsize=(18, 14))
axes = axes.flatten()

# -----------------------------
# Panel A: Age histogram
# -----------------------------
ax = axes[0]
bins = np.arange(np.floor(min_age / 5) * 5, np.ceil(max_age / 5) * 5 + 5, 5)

ax.hist(age, bins=bins, color=blue, edgecolor="white")
ax.axvline(mean_age, color=red, linestyle="--", linewidth=2, label=f"Mean = {mean_age:.1f}")
ax.axvline(median_age, color=gray, linestyle=":", linewidth=2, label=f"Median = {median_age:.1f}")

ax.set_title("A. Age distribution", loc="left", fontweight="bold")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Number of participants")
ax.grid(True, axis="y", alpha=0.25, color="0.75")
ax.legend(frameon=False)

style_axis(ax)

# -----------------------------
# Panel B: Gender bar plot
# -----------------------------
ax = axes[1]

plot_labels = []
plot_counts = []
plot_colors = []

if female_n > 0:
    plot_labels.append("Female")
    plot_counts.append(female_n)
    plot_colors.append(red)

if male_n > 0:
    plot_labels.append("Male")
    plot_counts.append(male_n)
    plot_colors.append(blue)

if other_n > 0:
    plot_labels.append("Other/Unknown")
    plot_counts.append(other_n)
    plot_colors.append(gray)

bars = ax.bar(plot_labels, plot_counts, color=plot_colors, edgecolor="black", linewidth=0.8)

ax.set_title("B. Gender distribution", loc="left", fontweight="bold")
ax.set_xlabel("Gender")
ax.set_ylabel("Number of participants")
ax.grid(True, axis="y", alpha=0.25, color="0.75")

for bar, count in zip(bars, plot_counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{count}",
        ha="center",
        va="bottom",
        fontsize=11,
        color="black"
    )

style_axis(ax)

# -----------------------------
# Panel C: Education level bar plot
# -----------------------------
ax = axes[2]

edu_labels = education_counts.index.tolist()
edu_values = education_counts.values.tolist()

bars = ax.bar(edu_labels, edu_values, color=blue, edgecolor="black", linewidth=0.8)

ax.set_title("C. Education level", loc="left", fontweight="bold")
ax.set_xlabel("Education")
ax.set_ylabel("Number of participants")
ax.grid(True, axis="y", alpha=0.25, color="0.75")
ax.tick_params(axis="x", rotation=30)

for bar, count in zip(bars, edu_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{count}",
        ha="center",
        va="bottom",
        fontsize=10,
        color="black"
    )

style_axis(ax)

# -----------------------------
# Panel D: Age by gender boxplot
# -----------------------------
ax = axes[3]

box_data = []
tick_labels = []
box_colors = []

for label, color in [("Female", red), ("Male", blue), ("Other/Unknown", gray)]:
    vals = df.loc[df["GenderLabel"] == label, "Age"].dropna().values
    if len(vals) > 0:
        box_data.append(vals)
        tick_labels.append(label)
        box_colors.append(color)

bp = ax.boxplot(
    box_data,
    tick_labels=tick_labels,
    patch_artist=True,
    widths=0.45
)

for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.45)
    patch.set_edgecolor("black")

for median in bp["medians"]:
    median.set_color("black")
    median.set_linewidth(2)

for whisker in bp["whiskers"]:
    whisker.set_color("black")

for cap in bp["caps"]:
    cap.set_color("black")

for flier in bp["fliers"]:
    flier.set_markeredgecolor("black")
    flier.set_markerfacecolor("black")
    flier.set_alpha(0.5)

rng = np.random.default_rng(42)
for i, vals in enumerate(box_data, start=1):
    x_jitter = rng.normal(i, 0.05, size=len(vals))
    ax.scatter(x_jitter, vals, s=18, alpha=0.5, color="black")

ax.set_title("D. Age by gender", loc="left", fontweight="bold")
ax.set_xlabel("Gender")
ax.set_ylabel("Age (years)")
ax.grid(True, axis="y", alpha=0.25, color="0.75")

# ---------------------------------------------------------
# Panel E. Total kept rows per stimulus
# ---------------------------------------------------------
ax = axes[4]

bars = ax.bar(
    stim_kept_sorted["task_label"],
    stim_kept_sorted["kept_rows"],
    color=blue,
    edgecolor="black",
    linewidth=0.6
)

ax.set_title("E. Total kept rows per stimulus", loc="left", fontweight="bold")
ax.set_xlabel("Cognitive task")
ax.set_ylabel("Kept rows")
ax.grid(True, axis="y", alpha=0.25, color="0.75")
ax.tick_params(axis="x", rotation=90)

style_axis(ax)

# ---------------------------------------------------------
# Panel F. Discarded percent by quality per stimulus
# ---------------------------------------------------------
ax = axes[5]

bars = ax.bar(
    stim_disc_sorted["task_label"],
    stim_disc_sorted["discarded_percent"],
    color=red,
    edgecolor="black",
    linewidth=0.6
)

ax.set_title("F. Discarded percent by quality", loc="left", fontweight="bold")
ax.set_xlabel("Cognitive task")
ax.set_ylabel("Discarded (%)")
ax.grid(True, axis="y", alpha=0.25, color="0.75")
ax.tick_params(axis="x", rotation=90)

style_axis(ax)

plt.tight_layout()
# -----------------------------
# Save
# -----------------------------
pdf_path = "MainFigures/F2/F2.pdf"
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
plt.close()

In [62]:
from matplotlib.colors import LinearSegmentedColormap
import json
from sklearn.decomposition import PCA

'''

FIGURE 3
'''

json_path = Path("FindOptimalProbForAugmentation/contrastive_hyperparams.json")

with open(json_path, "r", encoding="utf-8") as f:
    raw = json.load(f)

# =========================================================
# 2. JSON -> DATAFRAME
# =========================================================
df = pd.DataFrame(raw).T.reset_index().rename(columns={"index": "channel"})

# keep a stable channel order
channel_order = [
    "EEG.AF3", "EEG.F7", "EEG.F3", "EEG.FC5", "EEG.T7", "EEG.P7", "EEG.O1",
    "EEG.O2", "EEG.P8", "EEG.T8", "EEG.FC6", "EEG.F4", "EEG.F8", "EEG.AF4"
]

df = df[df["channel"].isin(channel_order)].copy()
df["channel"] = pd.Categorical(df["channel"], categories=channel_order, ordered=True)
df = df.sort_values("channel").reset_index(drop=True)

df["channel_short"] = df["channel"].map(lambda x: str(x).replace("EEG.", ""))

# force numeric types
numeric_cols = [
    "hidden_size", "batch_size", "projection_size", "temp", "num_layers_gru",
    "kernel_size", "stride", "padding", "channels"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# =========================================================
# 3. DERIVED METRICS
# =========================================================
df["conv_capacity"] = (df["channels"] * df["kernel_size"]) #+ (df["channels"]*df["kernel_size2"])
df["rnn_capacity"] = df["hidden_size"] * df["num_layers_gru"]
df["effective_rf"] = df["kernel_size"] * df["stride"]
df["compression"] = df["stride"] #* df["stride2"]
df["temporal_scale"] = df["kernel_size"] #+ df["kernel_size2"]
df["conv_to_rnn_ratio"] = df["conv_capacity"] / df["rnn_capacity"]


# =========================================================
# 5. MATRIX FOR HEATMAP AND PCA
# =========================================================
feature_cols = [
    "hidden_size",
    #"projection_size",
    "temp",
    "num_layers_gru",
    "kernel_size",
    "stride",
    "padding",
    "channels"
]

X = df[feature_cols].astype(float).copy()
Xz = (X - X.mean(axis=0)) / X.std(axis=0, ddof=0)
Xz = Xz.fillna(0.0)

pca = PCA(n_components=2)
pcs = pca.fit_transform(Xz.values)
df["PC1"] = pcs[:, 0]
df["PC2"] = pcs[:, 1]

# =========================================================
# 6. LEFT-RIGHT PAIRS
# =========================================================
pairs = [
    ("AF3", "AF4"),
    ("F7", "F8"),
    ("F3", "F4"),
    ("FC5", "FC6"),
    ("T7", "T8"),
    ("P7", "P8"),
    ("O1", "O2"),
]

pair_rows = []
for left, right in pairs:
    left_row = df.loc[df["channel_short"] == left].iloc[0]
    right_row = df.loc[df["channel_short"] == right].iloc[0]
    pair_rows.append({
        "pair": f"{left} vs {right}",
        "left_projection": left_row["projection_size"],
        "right_projection": right_row["projection_size"],
        "left_channels": left_row["channels"],
        "right_channels": right_row["channels"],
    })
pairs_df = pd.DataFrame(pair_rows)

# =========================================================
# 7. STYLE
# =========================================================
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "text.color": "black",
})

blue = "#123B7A"
red = "#C62828"
gray = "#6E6E6E"

def style_axis(ax):
    ax.tick_params(axis="both", colors="black", labelcolor="black")
    ax.xaxis.label.set_color("black")
    ax.yaxis.label.set_color("black")
    ax.title.set_color("black")
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.0)

# =========================================================
# 8. FIGURE
# =========================================================
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes = axes.flatten()

# ---------------------------------------------------------
# Panel A. Heatmap
# ---------------------------------------------------------

blue = "#123B7A"
red = "#C62828"



# ---------------------------------------------------------
# Panel A. Conv channels and projection size (dual barplot)
# ---------------------------------------------------------
ax = axes[0]
x = np.arange(len(df))
labels = df["channel_short"].tolist()

conv = df["channels"].to_numpy(dtype=float)
proj = df["projection_size"].to_numpy(dtype=float)

width = 0.4

# LEFT axis (blue) → conv channels
bars1 = ax.bar(
    x - width/2,
    conv,
    width=width,
    color=blue,
    edgecolor="black",
    linewidth=0.8,
    label="Conv channels"
)

ax.set_ylabel("Conv channels", color="black")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=0, ha="right")
ax.grid(True, axis="y", alpha=0.25)

# RIGHT axis (red) → projection size
ax2 = ax.twinx()

bars2 = ax2.bar(
    x + width/2,
    proj,
    width=width,
    color=red,
    edgecolor="black",
    linewidth=0.8,
    label="Projection size"
)

proj_mean = proj.mean()
proj_median = np.median(proj)
print(proj_median)

line_median = ax2.axhline(
    proj_median,
    color=red,
    linestyle="--",
    linewidth=1.5,
    label="Proj median"
)

ax2.set_ylabel("Projection size", color="black")
ax2.tick_params(axis="y", colors="black")
ax2.grid(False)
# Title
ax.set_title("A. Conv channels and projection size", loc="left", fontweight="bold")

# Legend (merge both)
handles1, labels1 = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

legend = ax.legend(
    handles1 + handles2,
    labels1 + labels2,
    frameon=True,
    loc="upper left"
)

# force black legend text
for text in legend.get_texts():
    text.set_color("black")

# Style
style_axis(ax)
for spine in ax2.spines.values():
    spine.set_color("black")
# ---------------------------------------------------------
# Panel B. Conv vs recurrent capacity
# ---------------------------------------------------------
ax = axes[1]
rf = df["effective_rf"].to_numpy(dtype=float)
# normalize
rf_norm = (rf - rf.min()) / (rf.max() - rf.min())
# scale to reasonable marker sizes (area!)
sizes = 50 + 400 * np.sqrt(rf_norm)

ax.scatter(
    df["conv_capacity"].to_numpy(dtype=float),
    df["rnn_capacity"].to_numpy(dtype=float),
    s=sizes,
    color=blue,
    edgecolors="black",
    linewidths=0.8
)

for _, row in df.iterrows():
    ax.text(float(row["conv_capacity"]) + 15, float(row["rnn_capacity"]) + 5.8, row["channel_short"], fontsize=8)

ax.set_title("B. Convolution versus recurrent capacity", loc="left", fontweight="bold")
ax.set_xlabel("Conv. capacity = (KernelSize × Channels)")
ax.set_ylabel("RNN capacity = Hidden size × GRU layers")
ax.grid(True, alpha=0.25)
rf_values = np.linspace(rf.min(), rf.max(), 3)
rf_values_norm = (rf_values - rf.min()) / (rf.max() - rf.min())
rf_sizes = 50 + 400 * np.sqrt(rf_values_norm)

handles = [plt.scatter([], [], s=s, color=blue, edgecolors="black") for s in rf_sizes]
labels = [f"{int(v)}" for v in rf_values]

ax.legend(handles, labels, title="Effective RF", loc="upper right", frameon=True)
ax.axvline(df["conv_capacity"].median(), color=gray, linestyle="--", linewidth=1)
ax.axhline(df["rnn_capacity"].median(), color=gray, linestyle="--", linewidth=1)
style_axis(ax)



# ---------------------------------------------------------
# Panel C. Kernel sizes and stride pattern
# ---------------------------------------------------------
ax = axes[2]

xpos = np.arange(len(df))
ratio = df["conv_to_rnn_ratio"].to_numpy(dtype=float)

bars = ax.bar(
    xpos,
    ratio,
    color=blue,
    edgecolor="black",
    linewidth=0.8
)

ax.set_title("C. Convolution to recurrent capacity ratio", loc="left", fontweight="bold")
ax.set_xlabel("Channel")
ax.set_ylabel("Conv / RNN ratio")
ax.set_xticks(xpos)
ax.set_xticklabels(df["channel_short"].tolist(), rotation=0, ha="right")
ax.grid(True, axis="y", color="0.8", linewidth=0.8)
ax.set_axisbelow(True)



style_axis(ax)
# ---------------------------------------------------------
# Panel D PCA biplot
# ---------------------------------------------------------

ax = axes[3]

pc1 = df["PC1"].to_numpy(dtype=float)
pc2 = df["PC2"].to_numpy(dtype=float)

# scatter of channels
ax.scatter(
    pc1,
    pc2,
    s=85,
    color=blue,
    edgecolors="black",
    zorder=3
)

for _, row in df.iterrows():
    ax.text(
        float(row["PC1"]) + 0.03,
        float(row["PC2"]) + 0.03,
        row["channel_short"],
        fontsize=8
    )

# PCA loadings
loadings = pca.components_.T[:, :2]   # shape: [n_features, 2]

# scale arrows so they fit nicely into the score plot
score_scale_x = np.max(np.abs(pc1))
score_scale_y = np.max(np.abs(pc2))
arrow_scale = 0.9 * min(score_scale_x, score_scale_y)

for i, feat in enumerate(feature_cols):
    x_loading = loadings[i, 0] * arrow_scale
    y_loading = loadings[i, 1] * arrow_scale

    ax.arrow(
        0, 0,
        x_loading, y_loading,
        color=red,
        width=0.004,
        head_width=0.06,
        length_includes_head=True,
        alpha=0.85,
        zorder=2
    )

    ax.text(
        x_loading * 1.08,
        y_loading * 1.08,
        feat,
        color=red,
        fontsize=8,
        ha="center",
        va="center"
    )

# reference lines
ax.axhline(0, color=gray, linewidth=1, linestyle="--")
ax.axvline(0, color=gray, linewidth=1, linestyle="--")

explained = pca.explained_variance_ratio_ * 100
ax.set_title("D. PCA biplot of channel hyperparameter profiles", loc="left", fontweight="bold")
ax.set_xlabel(f"PC1 ({explained[0]:.1f}% var)")
ax.set_ylabel(f"PC2 ({explained[1]:.1f}% var)")
ax.grid(True, alpha=0.25)

style_axis(ax)



pdf_path = "MainFigures/F3/F3.pdf"

fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
plt.close()


17.0


In [52]:
    ""'''

FIGURE 3 Supplements
'''
# Load data
path = "FindOptimalProbForAugmentation/contrastive_optuna_results.csv"
df = pd.read_csv(path)

# Clean channel names for display
df["ChannelShort"] = df["Channel"].str.replace("EEG.", "", regex=False)

# Best = minimum objective
best_idx = df.groupby("Channel")["Score"].idxmin()
best_df = (
    df.loc[best_idx, [
        "Channel", "ChannelShort", "Trial", "Score", "HiddenSize", "ProjSize", "Temp",
        "KernelSize", "Padding", "Stride", "KernelSize2", "Padding2", "Stride2"
    ]]
    .sort_values("ChannelShort")
    .reset_index(drop=True)
)

channel_order = best_df["Channel"].tolist()

# Explicit white background
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
})

fig, axes = plt.subplots(4, 4, figsize=(18, 14), constrained_layout=True, facecolor="white")
axes = axes.flatten()

line_color = "#123B7A"   # dark blue
dot_color = "#C62828"    # red

# Channel panels
for i, channel in enumerate(channel_order):
    ax = axes[i]
    sub = df[df["Channel"] == channel].sort_values("Trial").copy()
    sub["cum_best"] = sub["Score"].cummin()

    # Trial points in red
    ax.scatter(sub["Trial"], sub["Score"], alpha=0.8, s=20, color=dot_color)
    # Running best in dark blue
    ax.plot(sub["Trial"], sub["cum_best"], linewidth=2.2, color=line_color)

    best_row = sub.loc[sub["Score"].idxmin()]
    ax.scatter([best_row["Trial"]], [best_row["Score"]], s=110, marker="*", color=dot_color)

    ax.set_title(channel.replace("EEG.", ""), color="black")
    ax.set_xlabel("Trial", color="black")
    ax.set_ylabel("Objective", color="black")
    ax.grid(True, alpha=0.25, color="0.7")
    ax.tick_params(colors="black")
    for spine in ax.spines.values():
        spine.set_color("black")

    txt = (
        f"P={int(best_row['ProjSize'])}\n"
        f"T={best_row['Temp']:.3f} \n"
        f"U={best_row['Uniformity']:.3f} \n"
        f"A={best_row['Alignment']:.3f}"
    )
    ax.text(
        0.03, 0.03, txt,
        transform=ax.transAxes,
        fontsize=8,
        va="bottom",
        ha="left",
        color="black",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="0.5", alpha=0.9)
    )



# Save outputs
pdf_path = "MainFigures/F3/contrastive_multipanel_figure.pdf"
fig.savefig(pdf_path, bbox_inches="tight")
plt.close(fig)



In [55]:
'''

FIGURE 3
'''
# Basic cleanup
df["ChannelShort"] = df["Channel"].astype(str).str.replace("EEG.", "", regex=False)

# Best trial per channel based on minimization objective
best_idx = df.groupby("Channel")["Score"].idxmin()
best_df = (
    df.loc[best_idx, [
        "Channel", "ChannelShort", "Trial", "Score", "Alignment", "Uniformity",
        "HiddenSize", "ProjSize", "Temp", "KernelSize"
    ]]
    .sort_values("ChannelShort")
    .reset_index(drop=True)
)

channel_order = best_df["Channel"].tolist()

# Global axis limits for comparability across panels
align_min, align_max = df["Alignment"].min(), df["Alignment"].max()
uni_min, uni_max = df["Uniformity"].min(), df["Uniformity"].max()

# Add small margins
align_pad = (align_max - align_min) * 0.05 if align_max > align_min else 0.05
uni_pad = (uni_max - uni_min) * 0.05 if uni_max > uni_min else 0.05

# White figure style
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "font.size": 10,
})

fig, axes = plt.subplots(4, 4, figsize=(18, 14), constrained_layout=True, facecolor="white")
axes = axes.flatten()

line_color = "#123B7A"   # dark blue
dot_color = "#C62828"    # red

# 14 panels: alignment vs uniformity for each channel
for i, channel in enumerate(channel_order):
    ax = axes[i]
    sub = df[df["Channel"] == channel].copy()
    best_row = sub.loc[sub["Score"].idxmin()]

    # All trials
    ax.scatter(
        sub["Alignment"],
        sub["Uniformity"],
        s=26,
        alpha=0.8,
        color=dot_color
    )

    # Best trial
    ax.scatter(
        [best_row["Alignment"]],
        [best_row["Uniformity"]],
        s=130,
        marker="*",
        color=line_color,
        zorder=5
    )

    # Optional guiding lines through best point
    ax.axvline(best_row["Alignment"], color=line_color, alpha=0.15, linewidth=1.2)
    ax.axhline(best_row["Uniformity"], color=line_color, alpha=0.15, linewidth=1.2)

    ax.set_title(channel.replace("EEG.", ""), color="black")
    ax.set_xlabel("Alignment", color="black")
    ax.set_ylabel("Uniformity", color="black")
    #ax.set_xlim(align_min - align_pad, align_max + align_pad)
    #ax.set_ylim(uni_min - uni_pad, uni_max + uni_pad)
    ax.grid(True, alpha=0.25, color="0.7")
    ax.tick_params(colors="black")
    for spine in ax.spines.values():
        spine.set_color("black")

    txt = (
        f"best trial={int(best_row['Trial'])}\n"
        f"A={best_row['Alignment']:.3f}\n"
        f"U={best_row['Uniformity']:.3f}\n"
        f"obj={best_row['Score']:.3f}\n"
        f"K={int(best_row['KernelSize'])}"
    )
    ax.text(
        0.03, 0.03, txt,
        transform=ax.transAxes,
        fontsize=8,
        va="bottom",
        ha="left",
        color="black",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="0.5", alpha=0.9)
    )

# Summary panel 15: best-trial alignment by channel
ax = axes[14]
align_plot = best_df.sort_values("Alignment", ascending=True)
ax.bar(align_plot["ChannelShort"], align_plot["Alignment"], color=line_color)
ax.set_title("Best-trial alignment by channel", color="black")
ax.set_xlabel("Channel", color="black")
ax.set_ylabel("Alignment", color="black")
ax.tick_params(axis="x", rotation=45, colors="black")
ax.tick_params(axis="y", colors="black")
ax.grid(True, axis="y", alpha=0.25, color="0.7")
for spine in ax.spines.values():
    spine.set_color("black")

# Summary panel 16: best-trial uniformity by channel
ax = axes[15]
uni_plot = best_df.sort_values("Uniformity", ascending=True)
ax.bar(uni_plot["ChannelShort"], uni_plot["Uniformity"], color=dot_color)
ax.set_title("Best-trial uniformity by channel", color="black")
ax.set_xlabel("Channel", color="black")
ax.set_ylabel("Uniformity", color="black")
ax.tick_params(axis="x", rotation=45, colors="black")
ax.tick_params(axis="y", colors="black")
ax.grid(True, axis="y", alpha=0.25, color="0.7")
for spine in ax.spines.values():
    spine.set_color("black")

fig.suptitle(
    "Alignment and uniformity across EEG channels\nRed = all trials, blue star = best trial by minimum objective",
    fontsize=16,
    color="black"
)

pdf_path = "MainFigures/F3/contrastive_alignment_uniformity_multipanel.pdf"
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
plt.close(fig)



In [61]:
'''

FIGURE 4

'''
import pandas as pd, re

paths = {
    "alignment": "MainFigures/F4/alignment.csv",
    "train_loss": "MainFigures/F4/contrastive_loss_train.csv",
    "valid_loss": "MainFigures/F4/contrsative_loss_valid.csv",
    "heuristic": "MainFigures/F4/heuristic_score_valid.csv",
    "uniformity": "MainFigures/F4/uniformity.csv",
}

for name, f in paths.items():
    df = pd.read_csv(f)
    print(name, df.shape)
    print(df.columns[:8].tolist())
    print("last cols:", df.columns[-5:].tolist())
    print()
dfs = {k: pd.read_csv(v) for k, v in paths.items()}

channel_order = [
    "EEG.AF3", "EEG.F7", "EEG.F3", "EEG.FC5", "EEG.T7", "EEG.P7", "EEG.O1",
    "EEG.O2", "EEG.P8", "EEG.T8", "EEG.FC6", "EEG.F4", "EEG.F8", "EEG.AF4"
]

short = {c: c.replace("EEG.", "") for c in channel_order}
pat = re.compile(r"Contrastive_Channel_(EEG\.[A-Z0-9]+)\s-\s")

def extract_metric_table(df):
    out = pd.DataFrame({"Step": df["Step"]})
    for c in df.columns:
        if "__" in c or c == "Step":
            continue
        m = pat.search(c)
        if m:
            out[m.group(1)] = pd.to_numeric(df[c], errors="coerce")
    cols = ["Step"] + [c for c in channel_order if c in out.columns]
    return out[cols]

metric_tables = {k: extract_metric_table(v) for k, v in dfs.items()}

# -----------------------------
# Styling
# -----------------------------
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

blue = "#123B7A"
red = "#C62828"
gray = "#6E6E6E"

def style_axis(ax):
    ax.tick_params(axis="both", colors="black", labelcolor="black")
    ax.xaxis.label.set_color("black")
    ax.yaxis.label.set_color("black")
    ax.title.set_color("black")
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.0)

def plot_metric_panel(ax, tbl, title, ylabel, lower_is_better=True, smooth_window=35, legend=False):
    channels = [c for c in channel_order if c in tbl.columns]
    steps = tbl["Step"].to_numpy(dtype=float)

    # smooth
    smoothed = pd.DataFrame({"Step": steps})
    for c in channels:
        smoothed[c] = (
            tbl[c]
            .rolling(window=smooth_window, center=True, min_periods=1)
            .mean()
        )

    # green target regions
    if ylabel == "Alignment":
        ax.axhspan(0.2, 0.4, color="lightgreen", alpha=0.25, zorder=0)
    elif ylabel == "Uniformity":
        ax.axhspan(-2.5, -1.5, color="lightgreen", alpha=0.25, zorder=0)

    # distinct but balanced colors
    cmap = plt.get_cmap("tab10")
    colors = [cmap(i % 10) for i in range(len(channels))]

    for i, c in enumerate(channels):
        vals = smoothed[c].to_numpy(dtype=float)
        ax.plot(
            steps,
            vals,
            linewidth=1.6,
            alpha=0.95,
            color=colors[i],
            label=c.replace("EEG.", "")
        )

        if lower_is_better:
            best_idx = int(np.nanargmin(vals))
        else:
            best_idx = int(np.nanargmax(vals))

        ax.scatter(
            [steps[best_idx]],
            [vals[best_idx]],
            color=colors[i],
            s=18,
            zorder=4
        )

    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.grid(True, axis="y", color="black", alpha=0.2, linewidth=0.8)
    ax.set_axisbelow(True)

    if legend:
        legend = ax.legend(
            frameon=False,
            fontsize=8,
            ncol=2,
            loc="upper right"
        )
        for text in legend.get_texts():
            text.set_color("black")

    style_axis(ax)
    return channels

# -----------------------------
# Figure
# -----------------------------
fig, axes = plt.subplots(2, 3, figsize=(12, 8), constrained_layout=True)
axes = axes.flatten()
plot_metric_panel(
    axes[0], metric_tables["train_loss"],
    "A. Training contrastive loss", "Train loss",
    lower_is_better=True, smooth_window=75, legend=False
)

plot_metric_panel(
    axes[1], metric_tables["valid_loss"],
    "B. Validation contrastive loss", "Validation loss",
    lower_is_better=True, smooth_window=75, legend=False
)

plot_metric_panel(
    axes[2], metric_tables["alignment"],
    "C. Validation alignment", "Alignment",
    lower_is_better=False, smooth_window=50, legend=False
)

plot_metric_panel(
    axes[3], metric_tables["uniformity"],
    "D. Validation uniformity", "Uniformity",
    lower_is_better=True, smooth_window=50, legend=False
)

plot_metric_panel(
    axes[4], metric_tables["heuristic"],
    "E. Validation heuristic score", "Heuristic score",
    lower_is_better=True, smooth_window=75, legend=False
)

# F summary of best heuristic score by channel
# ---------------------------------------------------------
# Panel F. Best validation heuristic score by channel
# ---------------------------------------------------------
ax = axes[5]

heur = metric_tables["heuristic"]
heur_channels = [c for c in channel_order if c in heur.columns]
steps = heur["Step"].to_numpy(dtype=float)

best_scores = []
best_epochs = []

for c in heur_channels:
    vals = heur[c].to_numpy(dtype=float)
    idx = int(np.nanargmin(vals))
    best_scores.append(vals[idx])
    best_epochs.append(steps[idx])

summary = pd.DataFrame({
    "channel": heur_channels,
    "channel_short": [short[c] for c in heur_channels],
    "best_score": best_scores,
    "best_epoch": best_epochs,
}).sort_values("best_score", ascending=True).reset_index(drop=True)

scores = summary["best_score"].to_numpy(dtype=float)
norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
colors = plt.cm.bwr(norm)

bars = ax.bar(
    summary["channel_short"],
    summary["best_score"],
    color="#123B7A" ,
    edgecolor="black",
    linewidth=0.8
)

ax.set_title("F. Best validation heuristic score by channel", loc="left", fontweight="bold")
ax.set_xlabel("Channel")
ax.set_ylabel("Best score")
ax.tick_params(axis="x", rotation=45)

ax.grid(True, axis="y", alpha=0.25)
ax.set_axisbelow(True)
# tighten y axis so differences are visible
ax.set_ylim(scores.min() * 0.98, scores.max() * 1.02)


style_axis(ax)

# one compact legend
handles, labels = axes[4].get_legend_handles_labels()
legend = fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    frameon=True,
    bbox_to_anchor=(0.52, 0.99)
)

# force text color to black
for text in legend.get_texts():
    text.set_color("black")

# optional: title color (if you add one later)
if legend.get_title() is not None:
    legend.get_title().set_color("black")

# optional: make sure frame is visible and clean
legend.get_frame().set_facecolor("white")
legend.get_frame().set_edgecolor("black")

fig.suptitle(
    "Training trajectories of channel-wise contrastive models",
    fontsize=16, fontweight="bold", y=1.02
)


pdf_path = "MainFigures/F4/contrastive_training_multipanel.pdf"
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
plt.close(fig)


alignment (1000, 31)
['Step', 'Contrastive_Channel_EEG.AF4 - Val/Align', 'Contrastive_Channel_EEG.AF4 - Val/Align__MIN', 'Contrastive_Channel_EEG.AF4 - Val/Align__MAX', 'Contrastive_Channel_EEG.F8 - Val/Align', 'Contrastive_Channel_EEG.F8 - Val/Align__MIN', 'Contrastive_Channel_EEG.F8 - Val/Align__MAX', 'Contrastive_Channel_EEG.F4 - Val/Align']
last cols: ['Contrastive_Channel_EEG.P7 - Val/Align__MIN', 'Contrastive_Channel_EEG.P7 - Val/Align__MAX', 'Contrastive_Channel_EEG.T7 - Val/Align', 'Contrastive_Channel_EEG.T7 - Val/Align__MIN', 'Contrastive_Channel_EEG.T7 - Val/Align__MAX']

train_loss (1000, 31)
['Step', 'Contrastive_Channel_EEG.AF4 - Train/Loss', 'Contrastive_Channel_EEG.AF4 - Train/Loss__MIN', 'Contrastive_Channel_EEG.AF4 - Train/Loss__MAX', 'Contrastive_Channel_EEG.F8 - Train/Loss', 'Contrastive_Channel_EEG.F8 - Train/Loss__MIN', 'Contrastive_Channel_EEG.F8 - Train/Loss__MAX', 'Contrastive_Channel_EEG.F4 - Train/Loss']
last cols: ['Contrastive_Channel_EEG.P7 - Train/Loss__M